In [0]:
from pyspark.sql import functions as F

episodios = spark.table("mvp_engdados.silver.episodes")
imdb = spark.table("mvp_engdados.silver.imdb")

In [0]:
imdb_para_analise = imdb.filter(
    ~(
        (F.col("series_id") == "big_bang_theory") &
        (F.lower(F.trim("title")) == "unaired pilot")
    )
)

episodios_por_serie = (
    episodios
    .groupBy("series_id")
    .agg(
        F.countDistinct("episode_num_overall").alias("episode_count"),
        F.countDistinct("season").alias("season_count"),
        F.min("original_air_date").alias("first_air_date"),
        F.max("original_air_date").alias("last_air_date"),
        F.round(F.avg("us_viewers"), 0).alias("avg_us_viewers")
    )
    .withColumn(
        "duration_days",
        F.datediff("last_air_date", "first_air_date")
    )
    .withColumn(
        "duration_years",
        F.round(F.col("duration_days") / F.lit(365.25), 2)
    )
)

notas_por_serie = (
    imdb_para_analise
    .groupBy("series_id")
    .agg(
        F.count("*").alias("imdb_episode_count"),
        F.round(F.avg("imdb_rating"), 2).alias("avg_imdb_rating")
    )
)

gold_series_metrics = episodios_por_serie.join(
    notas_por_serie,
    on="series_id",
    how="full_outer"
)

In [0]:
audiencia_por_temporada = (
    episodios
    .groupBy("series_id", "season")
    .agg(
        F.countDistinct("episode_num_overall").alias("episode_count"),
        F.round(F.avg("us_viewers"), 0).alias("avg_us_viewers")
    )
)

notas_por_temporada = (
    imdb_para_analise
    .groupBy("series_id", "season")
    .agg(
        F.count("*").alias("imdb_episode_count"),
        F.round(F.avg("imdb_rating"), 2).alias("avg_imdb_rating")
    )
)

gold_season_metrics = audiencia_por_temporada.join(
    notas_por_temporada,
    on=["series_id", "season"],
    how="full_outer"
)

In [0]:
(
    gold_series_metrics.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_engdados.gold.series_metrics")
)

(
    gold_season_metrics.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_engdados.gold.season_metrics")
)

In [0]:
display(
    spark.table("mvp_engdados.gold.series_metrics")
    .orderBy("series_id")
)

display(
    spark.table("mvp_engdados.gold.season_metrics")
    .orderBy("series_id", "season")
)

series_id,episode_count,season_count,first_air_date,last_air_date,avg_us_viewers,duration_days,duration_years,imdb_episode_count,avg_imdb_rating
big_bang_theory,279,12,2007-09-24,2019-05-16,1.4646308E7,4252,11.64,279,7.82
breaking_bad,62,5,2008-01-20,2013-09-29,2324386.0,2079,5.69,62,9.03
brooklyn_99,153,8,2013-09-17,2021-09-16,2620719.0,2921,8.0,153,8.13
friends,236,10,1994-09-22,2004-05-06,2.5390805E7,3514,9.62,235,8.42
game_of_thrones,73,8,2011-04-17,2019-05-19,6447808.0,2954,8.09,73,8.75
how_i_met_your_mother,208,9,2005-09-19,2014-03-31,9044183.0,3115,8.53,208,8.14
lost,121,6,2004-09-22,2010-05-23,1.4580992E7,2069,5.66,119,8.55
the_office,201,9,2005-03-24,2013-05-16,7358856.0,2975,8.15,188,8.22
the_walking_dead,177,11,2010-10-31,2022-11-20,8137966.0,4403,12.05,177,7.92


series_id,season,episode_count,avg_us_viewers,imdb_episode_count,avg_imdb_rating
big_bang_theory,1,17,8443529.0,17,8.14
big_bang_theory,2,23,1.0142609E7,23,8.24
big_bang_theory,3,23,1.4118696E7,23,8.26
big_bang_theory,4,24,1.2558333E7,24,8.11
big_bang_theory,5,24,1.488125E7,24,8.0
big_bang_theory,6,24,1.6799167E7,24,8.0
big_bang_theory,7,24,1.7695E7,24,7.9
big_bang_theory,8,24,1.74975E7,24,7.49
big_bang_theory,9,24,1.770875E7,24,7.58
big_bang_theory,10,24,1.6670833E7,24,7.41
